# Supervised Fine Tuning with Gemini 2.5 Flash for Article Summarization

| Authors |
| --- |
| [Erwin Huizenga](https://www.linkedin.com/in/erwinhuizenga/) |
| [Deepak Moonat](https://github.com/dmoonat) |
| [Safiuddin Khaja](https://github.com/Safikh) |

## Overview

**Gemini** is a family of generative AI models developed by Google DeepMind that is designed for multimodal use cases. The Gemini API gives you access to the various Gemini models, such as Gemini 2.5 Pro/Flash, Gemini 2.5/Flash, Gemini/Flash and more.

This notebook demonstrates how to fine-tune the Gemini 2.5 Flash generative model using the Vertex AI Supervised Tuning feature. Supervised Tuning allows you to use your own training data to further refine the base model's capabilities towards your specific tasks.

Supervised Tuning uses labeled examples to tune a model. Each example demonstrates the output you want from your text model during inference.

First, ensure your training data is of high quality, well-labeled, and directly relevant to the target task. This is crucial as low-quality data can adversely affect the performance and introduce bias in the fine-tuned model.
- Training: Experiment with different configurations to optimize the model's performance on the target task.
- Evaluation:
  - Metric: Choose appropriate evaluation metrics that accurately reflect the success of the fine-tuned model for your specific task
  - Evaluation Set: Use a separate set of data to evaluate the model's performance


Refer to public [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini-supervised-tuning) for more details.


<hr/>

Before running this notebook, ensure you have:

- A Google Cloud project: Provide your project ID in the `PROJECT_ID` variable.

- Authenticated your Colab environment: Run the authentication code block at the beginning.

- Prepared training data (Test with your own data or use the one in the notebook): Data should be formatted in JSONL with prompts and corresponding completions.

### Objective

In this tutorial, you will learn how to use `Vertex AI` to tune a `Gemini 2.5 Flash` model.


This tutorial uses the following Google Cloud ML services:

- `Vertex AI`


The steps performed include:

- Prepare and load the dataset
- Load the `gemini-2.5-flash` model
- Evaluate the model before tuning
- Tune the model.
  - This will automatically create a Vertex AI endpoint and deploy the model to it
- Make a prediction using tuned model
- Evaluate the model after tuning

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), and use the [Pricing
Calculator](https://cloud.google.com/products/calculator/)
to generate a cost estimate based on your projected usage.

## Wikilingua Dataset

The dataset includes article and summary pairs from WikiHow. It consists of  article-summary pairs in multiple languages. Refer to the following [github repository](https://github.com/esdurmus/Wikilingua) for more details.

For this notebook, we have picked `english` language dataset.

### Dataset Citation

```bibtex
@inproceedings{ladhak-wiki-2020,
    title={WikiLingua: A New Benchmark Dataset for Multilingual Abstractive Summarization},
    author={Faisal Ladhak, Esin Durmus, Claire Cardie and Kathleen McKeown},
    booktitle={Findings of EMNLP, 2020},
    year={2020}
}
```

## Getting Started

### Install Gen AI SDK and other required packages

The new Google Gen AI SDK provides a unified interface to Gemini through both the Gemini Developer API and the Gemini API on Vertex AI. With a few exceptions, code that runs on one platform will run on both. This means that you can prototype an application using the Developer API and then migrate the application to Vertex AI without rewriting your code.


In [2]:
!python -m pip install --upgrade --quiet google-genai google-cloud-aiplatform rouge_score plotly jsonlines


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


- If you are running this notebook in a local development environment:
  - Install the [Google Cloud SDK](https://cloud.google.com/sdk).
  - Obtain authentication credentials. Create local credentials by running the following command and following the oauth2 flow (read more about the command [here](https://cloud.google.com/sdk/gcloud/reference/beta/auth/application-default/login)):

    ```bash
    gcloud init


    gcloud auth application-default login
    ```

## Step1: Import Libraries

In [8]:
import time

# For data handling.
import jsonlines
import pandas as pd

# For visualization.
import plotly.graph_objects as go

# For fine tuning Gemini model.
import vertexai
from google import genai

# For extracting vertex experiment details.
from google.cloud import aiplatform
from google.cloud.aiplatform.metadata import context
from google.cloud.aiplatform.metadata import utils as metadata_utils
from google.genai import types
from plotly.subplots import make_subplots

# For evaluation metric computation.
from rouge_score import rouge_scorer
from tqdm import tqdm

## Step2: Set Google Cloud project information and initialize Vertex AI and Gen AI SDK

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).


In [9]:
PROJECT_ID = "bdc-trainings"  # @param {type:"string"}
REGION = "us-central1"  # @param {type:"string"}

In [10]:
vertexai.init(project=PROJECT_ID, location=REGION)

client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

## Step3: Create Dataset in correct format

The dataset used to tune a foundation model needs to include examples that align with the task that you want the model to perform. Structure your training dataset in a text-to-text format. Each record, or row, in the dataset contains the input text (also referred to as the prompt) which is paired with its expected output from the model. Supervised tuning uses the dataset to teach the model to mimic a behavior, or task, you need by giving it hundreds of examples that illustrate that behavior.

Your dataset size depends on the task, and follows the recommendation mentioned in the `Overview` section. The more examples you provide in your dataset, the better the results.

### Dataset format

Training data should be structured within a JSONL file located at a Google Cloud Storage (GCS) URI. Each line (or row) of the JSONL file must adhere to a specific schema: It should contain a `contents` array, with objects inside defining a `role` (either "user" for user input or "model" for model output) and `parts`, containing the input data. For example, a valid data row would look like this:


```json
{
  "contents": [
    {
      "role": "user", # This indicates input content
      "parts": [
        {
          "text": "How are you?"
        }
      ]
    },
    {
      "role": "model", # This indicates target content
      "parts": [ # text only
        {
          "text": "I am good, thank you!"
        }
      ]
    }
  ] #  ... repeat "user", "model" for multi turns.
}
```

Refer to the public [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini-supervised-tuning-prepare#about-datasets) for more details.

To run a tuning job, you need to upload one or more datasets to a Cloud Storage bucket. You can either create a new Cloud Storage bucket or use an existing one to store dataset files. The region of the bucket doesn't matter, but we recommend that you use a bucket that's in the same Google Cloud project where you plan to tune your model.

### Step3 [a]: Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.


In [11]:
# Provide a bucket name
BUCKET_NAME = "tredencebucket"  # @param {type:"string"}
BUCKET_URI = f"gs://{BUCKET_NAME}"

Only if your bucket doesn't already exist: Run the following cell to create your Cloud Storage bucket.


In [12]:
!gsutil mb -l {REGION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://tredencebucket/...


### Step3 [b]: Upload tuning data to Cloud Storage

- Data used in this notebook is present in the public Google Cloud Storage(GCS) bucket.
- It's in Gemini finetuning dataset format

In [13]:
!gsutil ls gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua

gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_test_samples.csv
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_train_samples.jsonl
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_val_samples.jsonl


In [28]:
!gsutil cp gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/* datasets/

Copying gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_test_samples.csv...
Copying gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_train_samples.jsonl...
Copying gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_val_samples.jsonl...
| [3 files][  1.6 MiB/  1.6 MiB]                                                
Operation completed over 3 objects/1.6 MiB.                                      


#### Convert Gemini tuning dataset to Gemini 2.5 tuning dataset format

In [29]:
def save_jsonlines(file, instances) -> None:
    """Saves a list of json instances to a jsonlines file."""
    with jsonlines.open(file, mode="w") as writer:
        writer.write_all(instances)

In [30]:
def create_tuning_samples(file_path):
    """Creates tuning samples from a file."""
    with jsonlines.open(file_path) as reader:
        instances = []
        for obj in reader:
            instance = []
            for content in obj["messages"]:
                instance.append(
                    {"role": content["role"], "parts": [{"text": content["content"]}]}
                )
            instances.append({"contents": instance})
    return instances

In [31]:
train_file = "datasets/sft_train_samples.jsonl"
train_instances = create_tuning_samples(train_file)
len(train_instances)

500

In [32]:
# save the training instances to jsonl file
save_jsonlines(train_file, train_instances)

In [33]:
val_file = "datasets/sft_val_samples.jsonl"
val_instances = create_tuning_samples(val_file)
len(val_instances)

100

In [34]:
# save the validation instances to jsonl file
save_jsonlines(val_file, val_instances)

In [35]:
# Copy the tuning and evaluation data to your bucket.
!gsutil cp {train_file} {BUCKET_URI}/train/
!gsutil cp {val_file} {BUCKET_URI}/val/

Copying file://datasets/sft_train_samples.jsonl [Content-Type=application/octet-stream]...
\ [1 files][  1.2 MiB/  1.2 MiB]                                                
Operation completed over 1 objects/1.2 MiB.                                      
Copying file://datasets/sft_val_samples.jsonl [Content-Type=application/octet-stream]...
- [1 files][231.8 KiB/231.8 KiB]                                                
Operation completed over 1 objects/231.8 KiB.                                    


### Step3 [c]: Test dataset

- It contains document text(`input_text`) and corresponding reference summary(`output_text`), which will be compared with the model generated summary

In [36]:
# Load the test dataset using pandas as it's in the csv format.
testing_data_path = "datasets/sft_test_samples.csv"
test_data = pd.read_csv(testing_data_path)
test_data.head()

,input_text,output_text
0,Hold your arm out flat in front of you with yo...,Squeeze a line of lotion onto the tops of both...
1,"As you continue playing, surviving becomes pai...",Make a Crock Pot for better food. Create an Al...
2,Go to https://www.4kdownload.com/products/prod...,Download the 4K Video Downloader setup file. I...
3,You should know that vaginoplasty can treat a ...,Consider the health of your bladder. Find a so...
4,If you want to gather data on the frequency of...,Gather data to be graphed. Choose your range b...


In [37]:
test_data.loc[0, "input_text"]

'Hold your arm out flat in front of you with your elbow bent. The top of your forearm should form a level surface. Apply a line of lotion from the back of your hand up your arm almost to the crease of your elbow. Squeeze lotion onto both forearms.  Do not rub the lotion into your arms, rather let it sit on your arm in the line you squeezed. You can use as much or as little lotion as you feel is necessary to cover your back completely. Bend your elbows and reach both of your arms behind you, placing the lotion covered forearms against your back. Depending on how flexible you are, this may hurt a little. It might be easier to place one arm behind your back at a time. If you have shoulder pain or are not very flexible, this method may not work well for you. Rub your forearms and the backs of your hands up and down your back like windshield wipers covering as much of your back as you can. You can use your left arm first to cover your left side and then place your right arm behind and use i

In [38]:
# Article summary stats
stats = test_data["output_text"].apply(len).describe()
stats

count    100.000000
mean     186.230000
std       92.788655
min       28.000000
25%      127.250000
50%      171.000000
75%      227.000000
max      577.000000
Name: output_text, dtype: float64

In [39]:
print(f"Total `{stats['count']}` test records")
print(f"Average length is `{stats['mean']}` and max is `{stats['max']}` characters")
print("\nConsidering 1 token = 4 chars")

# Get ceil value of the tokens required.
tokens = (stats["max"] / 4).__ceil__()
print(
    f"\nSet max_token_length = stats['max']/4 = {stats['max'] / 4} ~ {tokens} characters"
)
print(f"\nLet's keep output tokens upto `{tokens}`")

Total `100.0` test records
Average length is `186.23` and max is `577.0` characters

Considering 1 token = 4 chars

Set max_token_length = stats['max']/4 = 144.25 ~ 145 characters

Let's keep output tokens upto `145`


In [40]:
# Maximum number of tokens that can be generated in the response by the LLM.
# Experiment with this number to get optimal output.
max_output_tokens = tokens

## Step4: Initailize model

The following Gemini text model support supervised tuning:

* `gemini-2.5-flash`

In [41]:
base_model = "gemini-2.5-flash"

## Step5: Test the Gemini model

### Generation config

- Each call that you send to a model includes parameter values that control how the model generates a response. The model can generate different results for different parameter values
- <strong>Experiment</strong> with different parameter values to get the best values for the task

Refer to the following [link](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/adjust-parameter-values) for understanding different parameters

**Prompt** is a natural language request submitted to a language model to receive a response back

Some best practices include
  - Clearly communicate what content or information is most important
  - Structure the prompt:
    - Defining the role if using one. For example, You are an experienced UX designer at a top tech company
    - Include context and input data
    - Provide the instructions to the model
    - Add example(s) if you are using them

Refer to the following [link](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/prompt-design-strategies) for prompt design strategies.

Wikilingua data contains the following task prompt at the end of the article, `Provide a summary of the article in two or three sentences:`

In [42]:
test_doc = test_data.loc[0, "input_text"]

prompt = f"""
{test_doc}
"""

config = {
    "temperature": 0.1,
    "max_output_tokens": max_output_tokens,
}

response = client.models.generate_content(
    model=base_model,
    contents=prompt,
    config=config,
).text
print(response)

This method describes how


In [43]:
# Ground truth
test_data.loc[0, "output_text"]

'Squeeze a line of lotion onto the tops of both forearms and the backs of your hands. Place your arms behind your back. Move your arms in a windshield wiper motion.'

## Step6: Evaluation before model tuning

- Evaluate the Gemini model on the test dataset before tuning it on the training dataset.

In [44]:
# Convert the pandas dataframe to records (list of dictionaries).
corpus = test_data.to_dict(orient="records")
# Check number of records.
len(corpus)

100

### Evaluation metric

The type of metrics used for evaluation depends on the task that you are evaluating. The following table shows the supported tasks and the metrics used to evaluate each task:

| Task             | Metric(s)                     |
|-----------------|---------------------------------|
| Classification   | Micro-F1, Macro-F1, Per class F1 |
| Summarization    | ROUGE-L                         |
| Question Answering | Exact Match                     |
| Text Generation  | BLEU, ROUGE-L                   |


<br/>

Refer to this [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/models/evaluate-models) for metric based evaluation.

- **Recall-Oriented Understudy for Gisting Evaluation (ROUGE)**: A metric used to evaluate the quality of automatic summaries of text. It works by comparing a generated summary to a set of reference summaries created by humans.

Now you can take the candidate and reference to evaluate the performance. In this case, ROUGE will give you:

- `rouge-1`, which measures unigram overlap
- `rouge-2`, which measures bigram overlap
- `rouge-l`, which measures the longest common subsequence

#### *Recall vs. Precision*

**Recall**, meaning it prioritizes how much of the information in the reference summaries is captured in the generated summary.

**Precision**, which measures how much of the generated summary is relevant to the original text.

<strong>Alternate Evaluation method</strong>: Check out the [AutoSxS](https://cloud.google.com/vertex-ai/generative-ai/docs/models/side-by-side-eval) evaluation for automatic evaluation of the task.


In [45]:
# Create rouge_scorer object for evaluation
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

In [46]:
def run_evaluation(model, corpus: list[dict]) -> pd.DataFrame:
    """Runs evaluation for the given model and data.

    Args:
      model: The generation model.
      corpus: The test data.

    Returns:
      A pandas DataFrame containing the evaluation results.
    """
    records = []
    for item in tqdm(corpus):
        document = item.get("input_text")
        summary = item.get("output_text")

        # Catch any exception that occur during model evaluation.
        try:
            response = client.models.generate_content(
                model=model,
                contents=document,
                config=config,
            )

            # Check if response is generated by the model, if response is empty then continue to next item.
            if not (
                response
                and response.candidates
                and response.candidates[0].content.parts
            ):
                print(
                    f"\nModel has blocked the response for the document.\n Response: {response}\n Document: {document}"
                )
                continue

            # Calculates the ROUGE score for a given reference and generated summary.
            scores = scorer.score(target=summary, prediction=response.text)

            # Append the results to the records list
            records.append(
                {
                    "document": document,
                    "summary": summary,
                    "generated_summary": response.text,
                    "scores": scores,
                    "rouge1_precision": scores.get("rouge1").precision,
                    "rouge1_recall": scores.get("rouge1").recall,
                    "rouge1_fmeasure": scores.get("rouge1").fmeasure,
                    "rouge2_precision": scores.get("rouge2").precision,
                    "rouge2_recall": scores.get("rouge2").recall,
                    "rouge2_fmeasure": scores.get("rouge2").fmeasure,
                    "rougeL_precision": scores.get("rougeL").precision,
                    "rougeL_recall": scores.get("rougeL").recall,
                    "rougeL_fmeasure": scores.get("rougeL").fmeasure,
                }
            )
        except AttributeError as attr_err:
            print("Attribute Error:", attr_err)
            continue
        except Exception as err:
            print("Error:", err)
            continue
    return pd.DataFrame(records)

In [47]:
# Batch of test data.
corpus_batch = corpus[:100]

<div class="alert alert-block alert-warning">
<b>⚠️ It will take ~2 mins for the evaluation run on the provided batch. ⚠️</b>
</div>

In [48]:
# Run evaluation using loaded model and test data corpus
evaluation_df = run_evaluation(base_model, corpus_batch)

  9%|▉         | 9/100 [00:13<01:57,  1.29s/it]


Model has blocked the response for the document.
 Response: sdk_http_response=HttpResponse(
  headers=<dict len=9>
) candidates=[Candidate(
  content=Content(
    role='model'
  ),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>
)] create_time=datetime.datetime(2025, 10, 8, 8, 21, 23, 321086, tzinfo=TzInfo(UTC)) model_version='gemini-2.5-flash' prompt_feedback=None response_id='Ax_maL7ME-eh3NoP5O6p-AU' usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=139,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=139
    ),
  ],
  thoughts_token_count=144,
  total_token_count=283,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
) automatic_function_calling_history=[] parsed=None
 Document: For even baking, position the oven rack at the center of the oven and bake one sheet of cookies at a time. If baking two sheets, it is recommended that you space the racks so as to divide the oven into thirds,

 11%|█         | 11/100 [00:15<01:48,  1.22s/it]


Model has blocked the response for the document.
 Response: sdk_http_response=HttpResponse(
  headers=<dict len=9>
) candidates=[Candidate(
  content=Content(
    role='model'
  ),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>
)] create_time=datetime.datetime(2025, 10, 8, 8, 21, 25, 590432, tzinfo=TzInfo(UTC)) model_version='gemini-2.5-flash' prompt_feedback=None response_id='BR_maOCEJM3-698PnKuguA0' usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=527,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=527
    ),
  ],
  thoughts_token_count=144,
  total_token_count=671,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
) automatic_function_calling_history=[] parsed=None
 Document: Instead of using store-bought, commercially-made flypaper, make your own using ingredients that aren’t poisonous.  Use any type of paper cut into 2” strips of any length.  Punch a hole in one end of the strip 

 24%|██▍       | 24/100 [00:32<01:38,  1.30s/it]

...

 80%|████████  | 80/100 [01:48<00:25,  1.30s/it]


Model has blocked the response for the document.
 Response: sdk_http_response=HttpResponse(
  headers=<dict len=9>
) candidates=[Candidate(
  content=Content(
    role='model'
  ),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>
)] create_time=datetime.datetime(2025, 10, 8, 8, 22, 58, 467181, tzinfo=TzInfo(UTC)) model_version='gemini-2.5-flash' prompt_feedback=None response_id='Yh_maO3BHIG7nvgPxqbayQ0' usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=482,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=482
    ),
  ],
  thoughts_token_count=144,
  total_token_count=626,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
) automatic_function_calling_history=[] parsed=None
 Document: A gene is a piece of "genetic code" that determines a trait in a living organism – for example, eye color. But eye color can be blue, or brown, or various other colors. These variations of the same gene are ca

 90%|█████████ | 90/100 [02:02<00:13,  1.36s/it]


Model has blocked the response for the document.
 Response: sdk_http_response=HttpResponse(
  headers=<dict len=9>
) candidates=[Candidate(
  content=Content(
    role='model'
  ),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>
)] create_time=datetime.datetime(2025, 10, 8, 8, 23, 12, 146032, tzinfo=TzInfo(UTC)) model_version='gemini-2.5-flash' prompt_feedback=None response_id='cB_maPD0CNur3NoPkYPI2A4' usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=467,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=467
    ),
  ],
  thoughts_token_count=144,
  total_token_count=611,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
) automatic_function_calling_history=[] parsed=None
 Document: Just like you cannot play an Xbox One game on an original Xbox, changes in hardware and software prevent older or cheaper computers from playing some games. All games will be labeled with both the "Minimum Spe

100%|██████████| 100/100 [02:16<00:00,  1.37s/it]


In [49]:
evaluation_df.head()

,document,summary,generated_summary,scores,rouge1_precision,rouge1_recall,rouge1_fmeasure,rouge2_precision,rouge2_recall,rouge2_fmeasure,rougeL_precision,rougeL_recall,rougeL_fmeasure
0,Hold your arm out flat in front of you with yo...,Squeeze a line of lotion onto the tops of both...,This,"{'rouge1': (0.0, 0.0, 0.0), 'rouge2': (0.0, 0....",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,"As you continue playing, surviving becomes pai...",Make a Crock Pot for better food. Create an Al...,To survive and thrive,"{'rouge1': (0.25, 0.05555555555555555, 0.09090...",0.250000,0.055556,0.090909,0.000000,0.000000,0.000000,0.250000,0.055556,0.090909
2,Go to https://www.4kdownload.com/products/prod...,Download the 4K Video Downloader setup file. I...,This guide explains how to download and instal...,"{'rouge1': (0.43548387096774194, 0.45762711864...",0.435484,0.457627,0.446281,0.147541,0.155172,0.151261,0.258065,0.271186,0.264463
3,You should know that vaginoplasty can treat a ...,Consider the health of your bladder. Find a so...,Vaginoplasty can treat a,"{'rouge1': (0.25, 0.03571428571428571, 0.0625)...",0.250000,0.035714,0.062500,0.000000,0.000000,0.000000,0.250000,0.035714,0.062500
4,If you want to gather data on the frequency of...,Gather data to be graphed. Choose your range b...,Histograms are effective,"{'rouge1': (0.3333333333333333, 0.043478260869...",0.333333,0.043478,0.076923,0.000000,0.000000,0.000000,0.333333,0.043478,0.076923


In [50]:
evaluation_df_stats = evaluation_df.dropna().describe()

In [51]:
# Statistics of the evaluation dataframe.
evaluation_df_stats

,rouge1_precision,rouge1_recall,rouge1_fmeasure,rouge2_precision,rouge2_recall,rouge2_fmeasure,rougeL_precision,rougeL_recall,rougeL_fmeasure
count,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000
mean,0.286833,0.072462,0.090056,0.037282,0.013347,0.015064,0.256491,0.062126,0.077677
std,0.257811,0.109788,0.105285,0.136794,0.038132,0.042342,0.239177,0.090212,0.086711
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.250000,0.037037,0.063492,0.000000,0.000000,0.000000,0.250000,0.034483,0.058824
75%,0.454545,0.076923,0.125000,0.000000,0.000000,0.000000,0.363636,0.058824,0.095238
max,1.000000,0.576923,0.508475,1.000000,0.200000,0.200000,1.000000,0.461538,0.406780


In [52]:
print("Mean rougeL_precision is", evaluation_df_stats.rougeL_precision["mean"])

Mean rougeL_precision is 0.2564910845270243


## Step7: Fine-tune the Model

 - `source_model`: Specifies the base Gemini model version you want to fine-tune.
 - `train_dataset`: Path to your training data in JSONL format.

  *Optional parameters*
 - `validation_dataset`: If provided, this data is used to evaluate the model during tuning.
 - `tuned_model_display_name`: Display name for the tuned model.
 - `epochs`: The number of training epochs to run.
 - `learning_rate_multiplier`: A value to scale the learning rate during training.
 - `adapter_size` : Gemini 2.5 Flash supports Adapter length [1, 2, 4, 8], default value is 4.

**Note: The default hyperparameter settings are optimized for optimal performance based on rigorous testing and are recommended for initial use. Users may customize these parameters to address specific performance requirements.**

In [53]:
tuned_model_display_name = "tredenceGemini"  # @param {type:"string"}

training_dataset = {
    "gcs_uri": f"{BUCKET_URI}/train/sft_train_samples.jsonl",
}

validation_dataset = types.TuningValidationDataset(
    gcs_uri=f"{BUCKET_URI}/val/sft_val_samples.jsonl"
)

# Tune a model using `tune` method.
sft_tuning_job = client.tunings.tune(
    base_model=base_model,
    training_dataset=training_dataset,
    config=types.CreateTuningJobConfig(
        tuned_model_display_name=tuned_model_display_name,
        validation_dataset=validation_dataset,
    ),
)

/tmp/ipykernel_4412/392514198.py:12: ExperimentalWarning: The SDK's tuning implementation is experimental, and may change in future versions.
  sft_tuning_job = client.tunings.tune(


In [54]:
# Get the tuning job info.
tuning_job = client.tunings.get(name=sft_tuning_job.name)
tuning_job

TuningJob(
  base_model='gemini-2.5-flash',
  create_time=datetime.datetime(2025, 10, 8, 8, 23, 29, 593372, tzinfo=TzInfo(UTC)),
  name='projects/456822750436/locations/us-central1/tuningJobs/3927222111533268992',
  sdk_http_response=HttpResponse(
    headers=<dict len=9>
  ),
  state=<JobState.JOB_STATE_PENDING: 'JOB_STATE_PENDING'>,
  supervised_tuning_spec=SupervisedTuningSpec(
    training_dataset_uri='gs://tredencebucket/train/sft_train_samples.jsonl',
    validation_dataset_uri='gs://tredencebucket/val/sft_val_samples.jsonl'
  ),
  tuned_model_display_name='tredenceGemini',
  update_time=datetime.datetime(2025, 10, 8, 8, 23, 29, 593372, tzinfo=TzInfo(UTC))
)

**Note: Tuning time depends on several factors, such as training data size, number of epochs, learning rate multiplier, etc.**

<div class="alert alert-block alert-warning">
<b>⚠️ It will take ~15 mins for the model tuning job to complete on the provided dataset and set configurations/hyperparameters. ⚠️</b>
</div>

### [Optional] Cancel Tuning Job

- Uncomment the below code to cancel the tuning job

In [ ]:
## Cancel the tuning job
# tuning_job = client.tunings.cancel(name=sft_tuning_job.name)
# tuning_job

### Status Check

In [ ]:
%%time
# Wait for job completion

running_states = [
    "JOB_STATE_PENDING",
    "JOB_STATE_RUNNING",
]

while tuning_job.state.name in running_states:
    print(".", end="")
    tuning_job = client.tunings.get(name=tuning_job.name)
    time.sleep(10)
print()

In [ ]:
tuned_model = tuning_job.tuned_model.endpoint
experiment_name = tuning_job.experiment

print("Tuned model experiment", experiment_name)
print("Tuned model endpoint resource name:", tuned_model)

### Step7 [a]: Tuning and evaluation metrics

#### Model tuning metrics

- `/train_total_loss`: Loss for the tuning dataset at a training step.
- `/train_fraction_of_correct_next_step_preds`: The token accuracy at a training step. A single prediction consists of a sequence of tokens. This metric measures the accuracy of the predicted tokens when compared to the ground truth in the tuning dataset.
- `/train_num_predictions`: Number of predicted tokens at a training step

#### Model evaluation metrics:

- `/eval_total_loss`: Loss for the evaluation dataset at an evaluation step.
- `/eval_fraction_of_correct_next_step_preds`: The token accuracy at an evaluation step. A single prediction consists of a sequence of tokens. This metric measures the accuracy of the predicted tokens when compared to the ground truth in the evaluation dataset.
- `/eval_num_predictions`: Number of predicted tokens at an evaluation step.

The metrics visualizations are available after the model tuning job completes. If you don't specify a validation dataset when you create the tuning job, only the visualizations for the tuning metrics are available.


In [ ]:
# Locate Vertex AI Experiment and Vertex AI Experiment Run
experiment = aiplatform.Experiment(experiment_name=experiment_name)
filter_str = metadata_utils._make_filter_string(
    schema_title="system.ExperimentRun",
    parent_contexts=[experiment.resource_name],
)
experiment_run = context.Context.list(filter_str)[0]

In [ ]:
# Read data from Tensorboard
tensorboard_run_name = f"{experiment.get_backing_tensorboard_resource().resource_name}/experiments/{experiment.name}/runs/{experiment_run.name.replace(experiment.name, '')[1:]}"
tensorboard_run = aiplatform.TensorboardRun(tensorboard_run_name)
metrics = tensorboard_run.read_time_series_data()

In [ ]:
def get_metrics(metric: str = "/train_total_loss"):
    """Get metrics from Tensorboard.

    Args:
      metric: metric name, eg. /train_total_loss or /eval_total_loss.

    Returns:
      steps: list of steps.
      steps_loss: list of loss values.
    """
    loss_values = metrics[metric].values
    steps_loss = []
    steps = []
    for loss in loss_values:
        steps_loss.append(loss.scalar.value)
        steps.append(loss.step)
    return steps, steps_loss

In [ ]:
# Get Train and Eval Loss
train_loss = get_metrics(metric="/train_total_loss")
eval_loss = get_metrics(metric="/eval_total_loss")

### Step7 [b]: Plot the metrics

In [ ]:
# Plot the train and eval loss metrics using Plotly python library

fig = make_subplots(
    rows=1, cols=2, shared_xaxes=True, subplot_titles=("Train Loss", "Eval Loss")
)

# Add traces
fig.add_trace(
    go.Scatter(x=train_loss[0], y=train_loss[1], name="Train Loss", mode="lines"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=eval_loss[0], y=eval_loss[1], name="Eval Loss", mode="lines"),
    row=1,
    col=2,
)

# Add figure title
fig.update_layout(title="Train and Eval Loss", xaxis_title="Steps", yaxis_title="Loss")

# Set x-axis title
fig.update_xaxes(title_text="Steps")

# Set y-axes titles
fig.update_yaxes(title_text="Loss")

# Show plot
fig.show()

## Step8: Load the Tuned Model

 - Load the fine-tuned model using `GenerativeModel` class with the tuning job model endpoint name.

 - Test the tuned model with the following prompt

In [ ]:
prompt

In [ ]:
if True:
    # Test with the loaded model.
    print("***Testing***")
    print(
        client.models.generate_content(
            model=tuned_model, contents=prompt, config=config
        ).text
    )
else:
    print("State:", tuning_job.state.name.state)
    print("Error:", tuning_job.state.name.error)

- We can clearly see the difference between summary generated pre and post tuning, as tuned summary is more inline with the ground truth format (**Note**: Pre and Post outputs, might vary based on the set parameters.)

  - *Pre*: `This article describes a method for applying lotion to your back using your forearms as applicators. By squeezing lotion onto your forearms and then reaching behind your back, you can use a windshield wiper motion to spread the lotion across your back. The method acknowledges potential limitations for those with shoulder pain or limited flexibility.`
  - *Post*: `Squeeze a line of lotion on your forearm. Reach behind you and rub your back.`
  - *Ground Truth*:` Squeeze a line of lotion onto the tops of both forearms and the backs of your hands. Place your arms behind your back. Move your arms in a windshield wiper motion.`

## Step9: Evaluation post model tuning

<div class="alert alert-block alert-warning">
<b>⚠️ It will take ~5 mins for the evaluation on the provided batch. ⚠️</b>
</div>

In [ ]:
# run evaluation
evaluation_df_post_tuning = run_evaluation(tuned_model, corpus_batch)

In [ ]:
evaluation_df_post_tuning.head()

In [ ]:
evaluation_df_post_tuning_stats = evaluation_df_post_tuning.dropna().describe()

In [ ]:
# Statistics of the evaluation dataframe post model tuning.
evaluation_df_post_tuning_stats

In [ ]:
print(
    "Mean rougeL_precision is", evaluation_df_post_tuning_stats.rougeL_precision["mean"]
)

#### Improvement

In [ ]:
improvement = round(
    (
        (
            evaluation_df_post_tuning_stats.rougeL_precision["mean"]
            - evaluation_df_stats.rougeL_precision["mean"]
        )
        / evaluation_df_stats.rougeL_precision["mean"]
    )
    * 100,
    2,
)
print(
    f"Model tuning has improved the rougeL_precision by {improvement}% (result might differ based on each tuning iteration)"
)

## Conclusion

Performance could be further improved:
- By adding more training samples. In general, improve your training data quality and/or quantity towards getting a more diverse and comprehensive dataset for your task
- By tuning the hyperparameters, such as epochs and learning rate multiplier
  - To find the optimal number of epochs for your dataset, we recommend experimenting with different values. While increasing epochs can lead to better performance, it's important to be mindful of overfitting, especially with smaller datasets. If you see signs of overfitting, reducing the number of epochs can help mitigate the issue
- You may try different prompt structures/formats and opt for the one with better performance

## Cleaning up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud
project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.


Otherwise, you can delete the individual resources you created in this tutorial.

Refer to this [instructions](https://cloud.google.com/vertex-ai/docs/tutorials/image-classification-custom/cleanup#delete_resources) to delete the resources from console.

In [ ]:
# Delete Experiment.
delete_experiments = True
if delete_experiments:
    experiments_list = aiplatform.Experiment.list()
    for experiment in experiments_list:
        if experiment.resource_name == experiment_name:
            print(experiment.resource_name)
            experiment.delete()
            break

print("***" * 10)

# Delete Endpoint.
delete_endpoint = True
# If force is set to True, all deployed models on this
# Endpoint will be first undeployed.
if delete_endpoint:
    for endpoint in aiplatform.Endpoint.list():
        if endpoint.resource_name == tuned_model:
            print(endpoint.resource_name)
            endpoint.delete(force=True)
            break

print("***" * 10)

# Delete Cloud Storage Bucket.
delete_bucket = True
if delete_bucket:
    ! gsutil -m rm -r $BUCKET_URI